# Non-local Wiener stage on Urban100 with the ADM ×2 pilot: S1 oracle gate, then A3 (CPU only)

BM3D's collaborative Wiener stage (block matching on a pilot, empirical Wiener shrinkage of the noisy image's 3-D
group spectra) adds image self-similarity that is independent of the diffusion prior's errors. Inputs: the
`urban100-tiles` outputs (ADM ×2 pilots of the whole image and of 512-px tiles, float16; the noisy images are
regenerated from the same seeds and checked against the saved sums). The pilot kind is the better of the two on
dev-12 at σ = 25 (amendment 1).

Pre-registered (decision of 2026-11-06), dev-12 = Urban100 images 8, 16, …, 96 for tuning, confirm-88 for claims:
* S1 gate: the Wiener stage with the *clean* image as pilot, blended λ·W + (1−λ)·ADM×2, on dev-12 at σ = 25. Kill the
  track if the best λ gains < +0.08 dB.
* A3 (only if S1 passes): the Wiener stage with the ADM ×2 pilot, one λ per σ chosen on dev-12, scored on
  confirm-88. Success: ≥ +0.05 dB at σ 25 and 50, Holm-corrected sign-flip p < 0.01, ≥ 65 % of images better.
  Kill: < +0.02 at both σ. If S1 fails, A3 runs on dev-12 only, as a labelled diagnostic.
Kaggle: CPU, internet on, input = kernel output `trnquanghuyn/urban100-tiles`.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bm3d"], check=True)

In [ ]:
import glob, inspect, io, json, math, os, re, time, urllib.request, zipfile
import bm3d
import numpy as np
import pandas as pd
import torch
from PIL import Image

SIGMAS = [15, 25, 50]
DEV12 = list(range(8, 100, 8))
LAMBDAS = [round(0.1 * k, 1) for k in range(11)]
SEED_SETS = ["McMaster", "Kodak24", "CBSD68", "Urban100"]
OUT, CACHE = "outputs", "/tmp/cache"
for d in (OUT, CACHE):
    os.makedirs(d, exist_ok=True)
T_START = time.time()
print("bm3d", getattr(bm3d, "__version__", "?"), inspect.signature(bm3d.bm3d_rgb), os.cpu_count(), "CPUs")

ROOT = sorted(glob.glob("/kaggle/input/**/pilots", recursive=True))
assert ROOT, "attach the output of trnquanghuyn/urban100-tiles"
ROOT = os.path.dirname(ROOT[0])
INFO = pd.read_csv(f"{ROOT}/image_info.csv")
G1 = pd.read_csv(f"{ROOT}/scores.csv")
dev25 = G1[(G1.sigma == 25) & G1.idx.isin(DEV12)].groupby("method").psnr.mean()
PILOT_KIND = max(("whole", "tiles-512"), key=lambda k: dev25.get(f"{k} ×2", -1.0))
print(ROOT, len(INFO), "jobs in the GPU run;", len(glob.glob(f"{ROOT}/pilots/*.npz")), "pilots;",
      "dev-12 σ25 ×2:", dev25.filter(like="×2").round(3).to_dict(), "→ pilot:", PILOT_KIND)

path = f"{CACHE}/Datasets.zip"
if not os.path.exists(path):
    urllib.request.urlretrieve("https://github.com/Li-Tong-621/DMID/releases/download/v1.0/Datasets.zip", path)


def natkey(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


with zipfile.ZipFile(path) as z:
    names = sorted([n for n in z.namelist() if n.startswith("Datasets/Gaussion/Urban100/") and not n.endswith("/")], key=natkey)
    TEST = [(n.split("/")[-1], np.asarray(Image.open(io.BytesIO(z.read(n))).convert("RGB"))) for n in names]


def noisy_of(clean_u8, sigma, idx):
    """Exactly the GPU run's noise (DMID protocol, Urban100 seeds)."""
    seed = 1_000_000 * (SEED_SETS.index("Urban100") + 1) + 1000 * sigma + idx
    x = torch.from_numpy(clean_u8).permute(2, 0, 1).float() / 255
    n = torch.randn(x.shape, generator=torch.Generator().manual_seed(seed))
    return x + (sigma / 255) * n / n.std(unbiased=False)


bad = []
for _, r in INFO.iterrows():
    s = float(noisy_of(TEST[int(r.idx)][1], int(r.sigma), int(r.idx)).double().sum())
    if abs(s - r.noisy_sum) > 1e-3 * max(1.0, abs(r.noisy_sum)) * 1e-3:
        bad.append((int(r.idx), int(r.sigma), s, r.noisy_sum))
print("noisy images regenerated; mismatches:", len(bad), bad[:3])
assert not bad, "noise regeneration differs from the GPU run"


def psnr(x_hwc, clean_u8):
    a = np.round(np.clip(x_hwc, 0, 1) * 255)
    return 10 * math.log10(255 ** 2 / np.mean((a - clean_u8.astype(np.float64)) ** 2))


def pilot(idx, sigma):
    return np.load(f"{ROOT}/pilots/{PILOT_KIND}_s{sigma}_{idx:03d}.npz")["x"].astype(np.float64).transpose(1, 2, 0)


def wiener_rgb(y, sigma01, guide):
    """bm3d_rgb's opponent-colour pipeline with the Wiener stage only, guided by `guide` (bm3d 4.x's bm3d_rgb takes no
    pilot): the noisy image and the pilot share one normalisation, σ is scaled per channel as bm3d_rgb scales it, and
    each opponent channel is filtered by the single-channel Wiener stage with its own 2-D pilot (bm4d's multichannel
    path rejects a 3-channel pilot)."""
    z, imax, imin, scale, a = bm3d.rgb_to(y, "opp")
    p = ((guide.reshape(-1, 3) @ a.T - imin) / (imax - imin)).reshape(guide.shape)
    sig = sigma01 * np.sqrt(scale)
    w = np.stack([np.squeeze(bm3d.bm3d(np.ascontiguousarray(z[..., c]), float(sig[c]), "np",
                                       stage_arg=np.ascontiguousarray(p[..., c]))) for c in range(3)], -1)
    return bm3d.rgb_to(w, "opp", True, imax, imin)[0]


def wiener_job(job):
    """One Wiener stage; returns per-λ PSNR of λ·W + (1−λ)·ADM×2."""
    idx, sigma, kind = job
    name, clean = TEST[idx]
    y = noisy_of(clean, sigma, idx).numpy().astype(np.float64).transpose(1, 2, 0)
    adm = pilot(idx, sigma)
    guide = clean.astype(np.float64) / 255 if kind == "clean pilot" else adm
    t0 = time.time()
    w = wiener_rgb(y, sigma / 255, guide)
    seconds = time.time() - t0
    return [dict(idx=idx, sigma=sigma, kind=kind, lam=lam, psnr=psnr(lam * w + (1 - lam) * adm, clean), seconds=seconds) for lam in LAMBDAS]


def run(jobs):
    """Sequential: the bm3d binary multithreads itself, and forked pools of it hung for 12 h in the first run."""
    rows = []
    for k, job in enumerate(jobs):
        rows += wiener_job(job)
        print(f"  {k + 1}/{len(jobs)} {job}: {rows[-1]['seconds']:.0f} s", flush=True)
    return pd.DataFrame(rows)

## 1 · S1 gate: clean-pilot Wiener stage blended with ADM ×2 (dev-12, σ = 25)

In [ ]:
t0 = time.time()
S1 = run([(i, 25, "clean pilot") for i in DEV12])
S1.to_csv(f"{OUT}/s1_gate.csv", index=False)
curve = S1.groupby("lam").psnr.mean()
base = float(curve.loc[0.0])
best_lam = float(curve.idxmax())
gate_gain = float(curve.max() - base)
S1_PASS = gate_gain >= 0.08
print(f"S1: ADM ×2 {base:.3f} dB; best λ {best_lam} gives {gate_gain:+.4f} dB → {'PASS' if S1_PASS else 'KILL the non-local track'}"
      f" ({(time.time() - t0) / 60:.1f} min, {S1.seconds.mean():.1f} s per Wiener stage)")
print(curve.round(4).to_string())
g1 = G1[(G1.method == f"{PILOT_KIND} ×2") & (G1.sigma == 25)].set_index("idx").psnr
gap = (S1[S1.lam == 0.0].set_index("idx").psnr - g1.reindex(DEV12)).abs().max()
print(f"λ = 0 reproduces the GPU run's ADM ×2 PSNR within {gap:.4f} dB (float16 pilots)")

## 2 · A3: ADM ×2 as the pilot; one λ per σ from dev-12; confirm-88 (or a dev-12 diagnostic if S1 failed)

In [ ]:
def sign_flip_p(d, n_flip=200_000, seed=0):
    rng = np.random.default_rng(seed)
    obs = abs(d.mean())
    signs = rng.choice([-1.0, 1.0], size=(n_flip, len(d)))
    return float((np.abs((signs * d).mean(1)) >= obs - 1e-12).mean())


t0 = time.time()
images = list(range(100)) if S1_PASS else DEV12
A3_SIGMAS = [25, 50]   # the pre-registered A3 criteria are at σ 25 and 50; σ 15 is skipped for CPU time
A3 = run([(i, s, "ADM ×2 pilot") for s in A3_SIGMAS for i in images])
A3["split"] = np.where(A3.idx.isin(DEV12), "dev", "confirm")
A3.to_csv(f"{OUT}/a3.csv", index=False)
results = []
for s in A3_SIGMAS:
    dev = A3[(A3.sigma == s) & (A3.split == "dev")].groupby("lam").psnr.mean()
    lam = float(dev.idxmax())
    part = A3[(A3.sigma == s) & (A3.split == ("confirm" if S1_PASS else "dev"))]
    wide = part.pivot_table(index="idx", columns="lam", values="psnr")
    d = (wide[lam] - wide[0.0]).to_numpy()
    results.append(dict(sigma=s, lam=lam, split="confirm-88" if S1_PASS else "dev-12 (diagnostic)", gain=float(d.mean()),
                        better=float((d > 0).mean()), p=sign_flip_p(d) if len(d) > 1 and lam > 0 else 1.0, n=len(d),
                        dev_gain=float(dev.max() - dev.loc[0.0])))
R = pd.DataFrame(results).sort_values("p")
R["p_holm"] = np.minimum(1.0, np.maximum.accumulate(R.p.to_numpy() * (len(R) - np.arange(len(R)))))
R = R.sort_values("sigma")
print(R.round(4).to_string(index=False), f"\n({(time.time() - t0) / 60:.1f} min)")
r25, r50 = R.set_index("sigma").loc[25], R.set_index("sigma").loc[50]
success = bool(S1_PASS and all(r.gain >= 0.05 and r.p_holm < 0.01 and r.better >= 0.65 for r in (r25, r50)))
kill = bool(r25.gain < 0.02 and r50.gain < 0.02)
print(f"A3 verdict: success {success}, kill {kill}" + ("" if S1_PASS else " (diagnostic only: S1 gate failed)"))
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(s1=dict(base=base, best_lam=best_lam, gain=gate_gain, passed=bool(S1_PASS)), a3=R.to_dict("records"),
                   a3_success=success, a3_kill=kill, minutes=(time.time() - T_START) / 60), f, indent=1, default=float)
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))